# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-2/ML-week-1-FLY/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** A page is worth reviewing for a CTR fix when it gets enough search exposure to trust the comparison (at least 500 impressions in the window, a real recorded position), and its click-through rate sits meaningfully below the median CTR of other pages at the *same position tier*. Rank candidates by how big that gap is, weighted by how much traffic is riding on the page — so a huge gap on a high-traffic page rises above the same-size gap on a barely-seen page.

**Reason code (one, since this is one rule):** `ctr_gap_vs_position_tier` — every ranked row carries this same code, because the whole baseline is this one comparison.

**Action label (one):** `review_title_meta_snippet` — the concrete thing an editor would do: rewrite the title tag, meta description, or on-SERP snippet to try to close the gap.

---

### Checking the two signals this rule leans on, before trusting them

**Signal A — CTR really does vary by position tier** (this is the signal directly behind FlyRank's CTR-fix flag from the session). Bucket table: median CTR and n per `position_tier`.

**Signal B — staleness (`days_since_last_update`) predicts CTR underperformance** (this is the signal behind the refresh flags from the session — I'm borrowing it to see if it should ALSO weigh into a CTR-focused rule, not just a refresh rule). Bucket table: underperform rate and n per `freshness_tier`.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Restrict to pages with enough exposure to trust any CTR comparison at all
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)].copy()

# --- Signal A: CTR by position_tier ---
signal_a = visible.groupby("position_tier").agg(n=("ctr", "size"), median_ctr=("ctr", "median")).sort_values("median_ctr", ascending=False)
print("SIGNAL A — median CTR by position_tier (n shown):")
print(signal_a)
print()
print("Verdict: MIXED.")
print("Direction mostly holds (striking > page_3_5 > deep, and the two big buckets dominate CTR),")
print("but top_3 (n=458) scores BELOW page_1 (n=7064) — the opposite of what you'd expect for the very")
print("best positions. The data dictionary itself warns the top_3 stratum has a low volume floor")
print("(median ~53 impressions/90d), so a handful of clicks swings its median CTR a lot. I'm treating")
print("position tier as real and usable for the bulk of pages, but NOT trusting the top_3 bucket's")
print("exact ranking without a bigger sample.")
print()

# --- Signal B: does staleness predict CTR underperformance? ---
tier_median = visible.groupby("position_tier")["ctr"].transform("median")
visible["underperform"] = (visible["ctr"] < tier_median).astype(int)

signal_b = visible.groupby("freshness_tier").agg(n=("underperform", "size"), underperform_rate=("underperform", "mean")).sort_values("underperform_rate", ascending=False)
print("SIGNAL B — underperform rate by freshness_tier (n shown):")
print(signal_b)
print()
print("Verdict: FALSE (for this specific use).")
print("The two large, trustworthy buckets — '91-180' (n=6558, 49.1%) and '0-30' (n=10063, 46.4%) —")
print("differ by only ~2.7 points. Freshly-updated pages underperform on CTR almost as often as older")
print("ones. The extreme-looking buckets ('31-90' at 61.4%, '181+' at 29.4%) both have tiny n (88 and")
print("17) and aren't reliable. Staleness clearly matters for FlyRank's refresh flag — but it does NOT")
print("meaningfully predict CTR-specific underperformance in this slice, so I'm leaving it OUT of this")
print("rule's score. That's a negative result that just kept my rule simpler and more honest.")


SIGNAL A — median CTR by position_tier (n shown):
                  n  median_ctr
position_tier                  
page_1         7064        0.24
top_3           458        0.20
striking       4485        0.17
page_3_5       4330        0.09
deep            389        0.00

Verdict: MIXED.
Direction mostly holds (striking > page_3_5 > deep, and the two big buckets dominate CTR),
but top_3 (n=458) scores BELOW page_1 (n=7064) — the opposite of what you'd expect for the very
best positions. The data dictionary itself warns the top_3 stratum has a low volume floor
(median ~53 impressions/90d), so a handful of clicks swings its median CTR a lot. I'm treating
position tier as real and usable for the bulk of pages, but NOT trusting the top_3 bucket's
exact ranking without a bigger sample.

SIGNAL B — underperform rate by freshness_tier (n shown):
                    n  underperform_rate
freshness_tier                          
31-90              88           0.613636
91-180           6558   

## 2. Build the ranked queue (writes the CSV)

`score = ctr_gap * impressions_90d` — readable on purpose, no fitted weights: a page's CTR shortfall below its position tier's median, multiplied by how much traffic is at stake. Bigger gap AND more traffic both push a page up the list; either one alone can still surface a page, but a page with both wins.

Staleness is deliberately left out of the score, per Signal B's honest negative above.


In [2]:
import os

visible["ctr_gap"] = tier_median - visible["ctr"]
visible["tier_median_ctr"] = tier_median
visible["score"] = visible["ctr_gap"] * visible["impressions_90d"]
visible["reason_code"] = "ctr_gap_vs_position_tier"
visible["action"] = "review_title_meta_snippet"

# Only rank pages that actually underperform (positive gap) — a negative gap means the page
# is already beating its tier's median, nothing to flag
queue = visible[visible["ctr_gap"] > 0].sort_values("score", ascending=False).reset_index(drop=True)

output_cols = ["content_id", "client_id", "position_tier", "avg_position", "impressions_90d",
               "ctr", "tier_median_ctr", "ctr_gap", "score", "reason_code", "action"]
queue_out = queue[output_cols].copy()

os.makedirs("work/outputs", exist_ok=True)
queue_out.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Ranked queue: {len(queue_out)} pages flagged for review (out of {len(visible)} visible pages checked)")
print(f"Written to work/outputs/baseline_action_score.csv")
queue_out.head(10)


Ranked queue: 7950 pages flagged for review (out of 16726 visible pages checked)
Written to work/outputs/baseline_action_score.csv


,content_id,client_id,position_tier,avg_position,impressions_90d,ctr,tier_median_ctr,ctr_gap,score,reason_code,action
0,content_36ff89c8214e,client_19581e27de,page_1,7.3,295097,0.05,0.24,0.19,56068.43,ctr_gap_vs_position_tier,review_title_meta_snippet
1,content_5fe46e04994d,client_4e07408562,page_1,4.2,517715,0.14,0.24,0.10,51771.50,ctr_gap_vs_position_tier,review_title_meta_snippet
2,content_c8e9d6ab9013,client_19581e27de,page_1,9.7,208678,0.00,0.24,0.24,50082.72,ctr_gap_vs_position_tier,review_title_meta_snippet
3,content_c84a0ab98e90,client_f369cb89fc,page_1,7.8,223271,0.03,0.24,0.21,46886.91,ctr_gap_vs_position_tier,review_title_meta_snippet
4,content_8451fc6f034d,client_d029fa3a95,top_3,2.3,272144,0.03,0.20,0.17,46264.48,ctr_gap_vs_position_tier,review_title_meta_snippet
5,content_453722754fea,client_f369cb89fc,page_1,7.6,140079,0.01,0.24,0.23,32218.17,ctr_gap_vs_position_tier,review_title_meta_snippet
6,content_73c54f78c06a,client_f369cb89fc,page_1,4.7,213963,0.10,0.24,0.14,29954.82,ctr_gap_vs_position_tier,review_title_meta_snippet
7,content_91652435f57a,client_19581e27de,page_1,7.8,159590,0.06,0.24,0.18,28726.20,ctr_gap_vs_position_tier,review_title_meta_snippet
8,content_c1fe78bc4e37,client_19581e27de,page_1,7.5,134055,0.03,0.24,0.21,28151.55,ctr_gap_vs_position_tier,review_title_meta_snippet
9,content_0919dd345d80,client_4e07408562,page_1,7.0,119217,0.02,0.24,0.22,26227.74,ctr_gap_vs_position_tier,review_title_meta_snippet


## 3. Top-10 review

For each of the top 10: what action it got, why it's there (in terms of the actual numbers), and what would make this specific pick wrong. Generated from the real ranked rows below, not written generically.


In [3]:
top10 = queue_out.head(10).reset_index(drop=True)

for i, row in top10.iterrows():
    print(f"#{i+1} — content_id={row['content_id']}")
    print(f"  Action: {row['action']}")
    print(f"  Why it's here: tier '{row['position_tier']}' (avg position {row['avg_position']:.1f}) has a "
          f"median CTR of {row['tier_median_ctr']:.2f}%, but this page only gets {row['ctr']:.2f}% "
          f"CTR from {int(row['impressions_90d'])} impressions — a gap of {row['ctr_gap']:.2f} points "
          f"on real traffic, giving it score {row['score']:.0f}.")
    print(f"  What would make this wrong: if this page's title/snippet is already accurate and the gap "
          f"is really explained by something the rule can't see — mismatched search intent, a stronger "
          f"competing result, or a seasonal dip in this specific topic that a title rewrite won't fix.")
    print()


#1 — content_id=content_36ff89c8214e
  Action: review_title_meta_snippet
  Why it's here: tier 'page_1' (avg position 7.3) has a median CTR of 0.24%, but this page only gets 0.05% CTR from 295097 impressions — a gap of 0.19 points on real traffic, giving it score 56068.
  What would make this wrong: if this page's title/snippet is already accurate and the gap is really explained by something the rule can't see — mismatched search intent, a stronger competing result, or a seasonal dip in this specific topic that a title rewrite won't fix.

#2 — content_id=content_5fe46e04994d
  Action: review_title_meta_snippet
  Why it's here: tier 'page_1' (avg position 4.2) has a median CTR of 0.24%, but this page only gets 0.14% CTR from 517715 impressions — a gap of 0.10 points on real traffic, giving it score 51771.
  What would make this wrong: if this page's title/snippet is already accurate and the gap is really explained by something the rule can't see — mismatched search intent, a stronger co

## 4. Weak picks + leakage check

**Weak pick check:** scanning the top 10 for any page where the gap looks more like noise than a real problem — specifically, any page whose `impressions_90d` sits close to the 500 minimum floor, where a handful of clicks could swing the CTR estimate a lot (the same volume-floor issue Signal A flagged for `top_3`).

**Leakage check:** confirm none of the following ever entered the score — `trend_direction`, `trend_pct` (label-source columns, per the data dictionary), any FlyRank product decision flag (none are shipped in this data), or anything from a future window (this dataset only has one trailing 90-day window, so there is no future window available to leak from in the first place).


In [4]:
# Weak pick check: any top-10 page close to the 500-impression floor?
near_floor = top10[top10["impressions_90d"] < 700]
print(f"Top-10 rows within 200 impressions of the 500 minimum floor: {len(near_floor)}")
if len(near_floor) > 0:
    print(near_floor[["content_id", "impressions_90d", "ctr_gap", "score"]])
    print("\nThese are the weakest picks in the top 10 — their CTR gap is real but built on the least")
    print("traffic, so it's the most likely to shrink or vanish with a few more weeks of data.")
else:
    print("None — every top-10 pick has comfortable volume well above the floor.")

print()

# Leakage check: confirm the score-building columns never touched label-source or excluded fields
score_inputs = {"impressions_90d", "avg_position", "ctr", "position_tier"}
banned_columns = {"trend_direction", "trend_pct"}  # label-source columns, per data dictionary
print(f"Columns used to build the score: {sorted(score_inputs)}")
print(f"Banned label-source columns touched: {sorted(score_inputs & banned_columns)}  (empty set = clean)")
print("No FlyRank product decision flags (health_score, priority_score, action_type) exist in this")
print("dataset at all, so there was nothing to accidentally include. This is a single trailing-90-day")
print("snapshot with no future window shipped, so a future-window leak isn't possible here by construction.")


Top-10 rows within 200 impressions of the 500 minimum floor: 0
None — every top-10 pick has comfortable volume well above the floor.

Columns used to build the score: ['avg_position', 'ctr', 'impressions_90d', 'position_tier']
Banned label-source columns touched: []  (empty set = clean)
No FlyRank product decision flags (health_score, priority_score, action_type) exist in this
dataset at all, so there was nothing to accidentally include. This is a single trailing-90-day
snapshot with no future window shipped, so a future-window leak isn't possible here by construction.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.